# C5.01 Corpus agenti - Tema 2


Scop: ne uităm la `data/corpus_typed.json`, verificăm distribuția bulelor și alegem ce exemple merită păstrate pentru vector store.



## 1. Setup

In [1]:
from pathlib import Path
import os
import pandas as pd

PROJECT_ROOT = Path(r"D:\Facultate\ADC\AI_Engineering\echochamber-project-team-1")
os.chdir(PROJECT_ROOT)
DATA_PATH = Path("data/typed/corpus_typed.jsonl")


In [2]:
DATA_PATH

WindowsPath('data/typed/corpus_typed.jsonl')

## 2. Încărcăm corpusul

In [3]:
df = pd.read_json(DATA_PATH, lines=True)
df.head(2)

,id,text,source_channel,channel_family,video_title,target_refined,stance_to_target,confidence,discourse_type,discourse_subtype,type_confidence
0,yt_Tx8GhU2LeyI_UgwoWOyzF2UbPYnguUB4AaABAg,Am toată încrederea că oameni ( de bine ) ca :...,AlephNewsOfficial,mainstream,ATENȚIE: România e „binevenită” să aplice iar ...,simion,anti,0.9,T3_opozitie_suveranista,opozitie_difuza,medium
1,yt_joXkZDqGZQU_Ugyqb1XZ7P8GTnJS_4p4AaABAg,Semneaza Bo$$ ca la urmatoarele alegerii nu ma...,NicusorDanRO,mainstream_actor,🟢 Declarații de presă comune cu Președintele U...,nicusor_dan,anti,0.9,T2_grievance_anti_sistem,grievance_mobilizator,medium


## 3. Vedem structura datelor

In [4]:
len(df)

896

In [5]:
print("Coloane:")
df.columns

Coloane:


Index(['id', 'text', 'source_channel', 'channel_family', 'video_title',
       'target_refined', 'stance_to_target', 'confidence', 'discourse_type',
       'discourse_subtype', 'type_confidence'],
      dtype='str')

In [6]:
df["discourse_type"].value_counts(dropna=False)

discourse_type
T3_opozitie_suveranista       150
T1_suport_personalist         150
T5_pro_democratic_european    150
T2_grievance_anti_sistem      149
T6_afectiv_pozitional         149
T4_conspiratie_externalism    148
Name: count, dtype: int64

In [26]:
## Exemplu de text pentru fiecare tip de discurs

In [7]:
for bubble in df["discourse_type"].value_counts().index:
    print("\n" + "="*80)
    print(bubble)
    print("="*80)
    
    sample = df[df["discourse_type"] == bubble]["text"].dropna().head(5)
    
    for i, text in enumerate(sample, 1):
        print(f"\n{i}. {text[:50]}")


T3_opozitie_suveranista

1. Am toată încrederea că oameni ( de bine ) ca : G S

2. Apropo de avalansa de troli ce se devarsa si acum 

3. Aceasta nu este o emisiune....este o regizare mize

4. La pregatit bine Putin a investit bani in Guru Geo

5. Eu am vorbit cu susținători de ai lui CG, îs deran

T1_suport_personalist

1. Ne au distrus hoții 😢,,,nu ne mai aparține nimic,,

2. Era si timpul sa treceti la atac, prea multa defen

3. Vă mulțumim și noi și copii noștri care lucrează î

4. 1:35 NOI știm că Sistemul ÎL hărțuiește mișelește 

5. Ideea că poporul are dreptul sau chiar datoria de 

T5_pro_democratic_european

1. Nu are Ce cauta pe teritoriulRomaniei, indiferent 

2. Tipul care și-a dat demisia în noiembrie la cum gâ

3. + la asta, d. Președinte, trebuie sa existe O MAJO

4. Mă bucur să văd un Președinte care se comportă fir

5. Deci exemplu asta cu Parisul cred ca e cea mai tar

T2_grievance_anti_sistem

1. Semneaza Bo$$ ca la urmatoarele alegerii nu mai ie

2. ESTE NEVOIE DE

# 4. Pastram doar 150 de bula

In [8]:
# Alegem un subset simplu pentru curs: maximum 150 texte per bulă

N_PER_BUBBLE = 150

df_sample = (
    df[df["discourse_type"].notna()]
    .sort_values("type_confidence", ascending=False)
    .groupby("discourse_type", group_keys=False)
    .head(N_PER_BUBBLE)
    .copy()
)

df_sample["discourse_type"].value_counts()

discourse_type
T3_opozitie_suveranista       150
T1_suport_personalist         150
T5_pro_democratic_european    150
T2_grievance_anti_sistem      149
T6_afectiv_pozitional         149
T4_conspiratie_externalism    148
Name: count, dtype: int64

In [9]:
# Păstrăm doar coloanele utile și salvăm corpusul pentru etapa următoare

keep_cols = [
    "id", "text", "source_channel", "channel_family", "video_title",
    "target_refined", "stance_to_target",
    "confidence", "discourse_type", "discourse_subtype", "type_confidence"
]

df_sample = df_sample[keep_cols].drop_duplicates(subset="text").copy()

OUT_PATH = Path("data/typed/corpus_c5_sample.jsonl")
df_sample.to_json(OUT_PATH, orient="records", lines=True, force_ascii=False)

print(df_sample.shape)
print(OUT_PATH)
df_sample.head(2)

(896, 11)
data\typed\corpus_c5_sample.jsonl


,id,text,source_channel,channel_family,video_title,target_refined,stance_to_target,confidence,discourse_type,discourse_subtype,type_confidence
0,yt_Tx8GhU2LeyI_UgwoWOyzF2UbPYnguUB4AaABAg,Am toată încrederea că oameni ( de bine ) ca :...,AlephNewsOfficial,mainstream,ATENȚIE: România e „binevenită” să aplice iar ...,simion,anti,0.9,T3_opozitie_suveranista,opozitie_difuza,medium
402,yt_opXUERp44N4_UgyRMEikJl6sUpuMCMp4AaABAg,"De coiful dacic, de colectiv, de dosarul revol...",CălinGeorgescu-CanalulOficial,sovereigntist,Călin Georgescu - Despăducherea și Revoluția I...,georgescu,unclear,0.9,T4_conspiratie_externalism,conspiratie_suveranista,medium


## 5. Exportul bulelor finale
În această etapă transformăm corpusul tipologizat în fișiere separate, câte unul pentru fiecare bulă finală.
Codurile `T1`, `T2`, `T3`, `T4`, `T5` vin din adnotare, dar în aplicație folosim numele finale ale agenților:
| Agent | Personalitate | Cum vorbește | Ce îl definește |
|---|---|---|---|
| Personalist-salvator | devotat, admirativ, sigur | laudativ, emoțional, încrezător | vede liderul ca soluție excepțională |
| Anti-sistem | furios, suspicios, dezamăgit | acuzator, moralizator, direct | vede instituțiile și „sistemul” ca profund compromise |
| Anti-suveranist | critic, vigilent, defensiv | contestatar, mai argumentativ | respinge liderii și discursul suveranist |
| Conspiraționist | alarmist, hiper-suspicios | speculativ, revelator, totalizant | explică evenimentele prin forțe ascunse și actori externi |
| Pro-european | normativ, moderat, legalist | sobru, justificativ, procedural | apără regulile, instituțiile și ancorarea europeană |
`T6_afectiv_pozitional` nu devine agent principal în C5. Rămâne categorie transversală / rezervă.

In [10]:
BUBBLES = {
    "T1_suport_personalist": {
        "agent": "Personalist-salvator",
        "slug": "personalist_salvator",
        "personality": "devotat, admirativ, sigur",
        "speaks": "laudativ, emoțional, încrezător",
        "definition": "vede liderul ca soluție excepțională",
    },
    "T2_grievance_anti_sistem": {
        "agent": "Anti-sistem",
        "slug": "anti_sistem",
        "personality": "furios, suspicios, dezamăgit",
        "speaks": "acuzator, moralizator, direct",
        "definition": "vede instituțiile și „sistemul” ca profund compromise",
    },
    "T3_opozitie_suveranista": {
        "agent": "Anti-suveranist",
        "slug": "anti_suveranist",
        "personality": "critic, vigilent, defensiv",
        "speaks": "contestatar, mai argumentativ",
        "definition": "respinge liderii și discursul suveranist",
    },
    "T4_conspiratie_externalism": {
        "agent": "Conspiraționist",
        "slug": "conspirationist",
        "personality": "alarmist, hiper-suspicios",
        "speaks": "speculativ, revelator, totalizant",
        "definition": "explică evenimentele prin forțe ascunse și actori externi",
    },
    "T5_pro_democratic_european": {
        "agent": "Pro-european",
        "slug": "pro_european",
        "personality": "normativ, moderat, legalist",
        "speaks": "sobru, justificativ, procedural",
        "definition": "apără regulile, instituțiile și ancorarea europeană",
    },
}

In [11]:
"""
OUT_DIR = Path("data/bubbles")
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_PER_BUBBLE = 50

for old_type, meta in BUBBLES.items():
    bubble_df = (
        df_sample[df_sample["discourse_type"] == old_type]
        .drop_duplicates(subset="text")
        .head(N_PER_BUBBLE)
        .copy()
    )

    bubble_df["agent"] = meta["agent"]
    bubble_df["slug"] = meta["slug"]
    bubble_df["personality"] = meta["personality"]
    bubble_df["speaks"] = meta["speaks"]
    bubble_df["definition"] = meta["definition"]
    bubble_df["source_type"] = old_type

    out_path = OUT_DIR / f"{meta['slug']}.jsonl"
    bubble_df.to_json(out_path, orient="records", lines=True, force_ascii=False)

    print(f"{meta['agent']}: {len(bubble_df)} texte -> {out_path}")

"""

'\nOUT_DIR = Path("data/bubbles")\nOUT_DIR.mkdir(parents=True, exist_ok=True)\n\nN_PER_BUBBLE = 50\n\nfor old_type, meta in BUBBLES.items():\n    bubble_df = (\n        df_sample[df_sample["discourse_type"] == old_type]\n        .drop_duplicates(subset="text")\n        .head(N_PER_BUBBLE)\n        .copy()\n    )\n\n    bubble_df["agent"] = meta["agent"]\n    bubble_df["slug"] = meta["slug"]\n    bubble_df["personality"] = meta["personality"]\n    bubble_df["speaks"] = meta["speaks"]\n    bubble_df["definition"] = meta["definition"]\n    bubble_df["source_type"] = old_type\n\n    out_path = OUT_DIR / f"{meta[\'slug\']}.jsonl"\n    bubble_df.to_json(out_path, orient="records", lines=True, force_ascii=False)\n\n    print(f"{meta[\'agent\']}: {len(bubble_df)} texte -> {out_path}")\n\n'

## 6. Alege agentul tău și verifică textele
Fiecare membru al echipei lucrează pe un singur agent.
Datele vin din `df_sample`, iar agentul este legat de eticheta tehnică `discourse_type`.
Scopul este să alegi aproximativ 50 texte bune pentru agentul tău.

In [12]:
# Alegeți un agent și încărcați textele corespunzătoare pentru etapa următoare
AGENTS = {
    "Personalist-salvator": {
        "type": "T1_suport_personalist",
        "slug": "personalist_salvator",
        "personality": "devotat, admirativ, sigur",
        "speaks": "laudativ, emoțional, încrezător",
        "definition": "vede liderul ca soluție excepțională",
    },
    "Anti-sistem": {
        "type": "T2_grievance_anti_sistem",
        "slug": "anti_sistem",
        "personality": "furios, suspicios, dezamăgit",
        "speaks": "acuzator, moralizator, direct",
        "definition": "vede instituțiile și „sistemul” ca profund compromise",
    },
    "Anti-suveranist": {
        "type": "T3_opozitie_suveranista",
        "slug": "anti_suveranist",
        "personality": "critic, vigilent, defensiv",
        "speaks": "contestatar, mai argumentativ",
        "definition": "respinge liderii și discursul suveranist",
    },
    "Conspiraționist": {
        "type": "T4_conspiratie_externalism",
        "slug": "conspirationist",
        "personality": "alarmist, hiper-suspicios",
        "speaks": "speculativ, revelator, totalizant",
        "definition": "explică evenimentele prin forțe ascunse și actori externi",
    },
    "Pro-european": {
        "type": "T5_pro_democratic_european",
        "slug": "pro_european",
        "personality": "normativ, moderat, legalist",
        "speaks": "sobru, justificativ, procedural",
        "definition": "apără regulile, instituțiile și ancorarea europeană",
    },
}

MY_AGENT = "Personalist-salvator"  # Alegeți unul dintre agenți: "Personalist-salvator", "Anti-sistem", "Anti-suveranist", "Conspiraționist", "Pro-european"

meta = AGENTS[MY_AGENT]

my_df = (
    df_sample[df_sample["discourse_type"] == meta["type"]]
    .drop_duplicates(subset="text")
    .copy()
)

print("Agent:", MY_AGENT)
print("Tip discurs:", meta["type"])
print("Texte disponibile:", len(my_df))

my_df[["id", "type_confidence", "discourse_subtype", "text"]].head(2)

Agent: Personalist-salvator
Tip discurs: T1_suport_personalist
Texte disponibile: 150


,id,type_confidence,discourse_subtype,text
386,yt_cF6iydQa9ss_UgyMxaKYDJ6_GP9NXXJ4AaABAg,medium,suport_afectiv_suveranist,"Eu sper ca va veni si ziua ""Justitiei Divina"" ..."
387,yt_5QYDDejaR5E_UgwOLigdAGsShcgE8Jp4AaABAg,medium,suport_afectiv_suveranist,Sunt patrioți și oameni onesti și in interioru...


### Cum verifici textele
Citește textele afișate mai jos.
Dacă un text este slab, copiază ID-ul lui în lista `REMOVE_IDS`.
Elimină texte prea scurte, duplicate, ambigue sau care nu exprimă clar vocea agentului.|

- pot sa folosesc un notepad pentru IDs
daca nu gasesti 50 de comentarii bune, incarca mai multe.
- poti folosi si alte metode (export text, csv) pentru vizualizarea si alegerea comentariilor

In [13]:
for _, row in my_df.head(70).iterrows():
    print("=" * 80)
    print("ID:", row["id"])
    print("Confidence:", row["type_confidence"])
    print("Subtype:", row["discourse_subtype"])
    print(row["text"][:700])

ID: yt_cF6iydQa9ss_UgyMxaKYDJ6_GP9NXXJ4AaABAg
Confidence: medium
Subtype: suport_afectiv_suveranist
Eu sper ca va veni si ziua "Justitiei Divina" care va face dreptate! De la astia nu ne putem astepta la dreptate din pacate!Rusine sa le fie pentru ce fac cu domnul C.Georgescu!
ID: yt_5QYDDejaR5E_UgwOLigdAGsShcgE8Jp4AaABAg
Confidence: medium
Subtype: suport_afectiv_suveranist
Sunt patrioți și oameni onesti și in interiorul Sistemului.Oameni care doresc Schimbarea.Ajung.Destul 35 de ani de corupție trădare minciunii manipulare.Nu mai tine Floriane .Poporul s-a trezit in conștiința .
ID: yt_oine49HHTXQ_Ugw57cLmGKlwkomjhG94AaABAg
Confidence: medium
Subtype: suport_afectiv_suveranist
Doamne ajuta AuR la putere ca SA scapam de hienele PSD, PNL, UDMR Si Sorosistii USR
ID: yt_oine49HHTXQ_Ugz3uUUbe6bmpTbfJFt4AaABAg
Confidence: medium
Subtype: suport_afectiv_suveranist
CUM AU OPRIT EI TURUL 2, AȘA OPRIM ȘI NOI TURUL 2, SIMION DIN PRIMUL TUR.
ID: yt_HDNif5_foi4_Ugwy-z6hrAqUylVVL3J4AaABAg
Confiden

In [14]:
# elimin textele slabe și păstrez 50

REMOVE_IDS = [
    # pune aici ID-urile textelor slabe

    # prea scurte
    "yt_4YPx5S_lPLg_UgyCPXLaPQuwf5bnksV4AaABAg",
    "yt_jvV4hfEOyQM_Ugx923FsNvV9TB41b_p4AaABAg",
    "yt_rG4DGen0GFw_Ugx7nWF1YB6EP1E75Gh4AaABAg",
    "yt_V12JqI7KjuQ_Ugw9gtj_YX1Cv9jrVAZ4AaABAg",
    "yt_wBTMhbcFAL0_Ugwfn6zVbpbXUuMKmKR4AaABAg",

    # ambiguu
    "yt_opXUERp44N4_UgwsU-M6kQ3KGopEBVt4AaABAg",
    "yt_y2972A4bDYg_UgwLQh0hCq4UQ9hcKBJ4AaABAg",
    "yt_y2972A4bDYg_UgwJ7oUwiCZNH4SuO4J4AaABAg",
    "yt_RyS5VvxNsbg_UgwC2Q1Gly07czFuchN4AaABAg",
    "yt_wBTMhbcFAL0_UgzIo3TgoYy_R4Fe7G54AaABAg",

    # incoerent
    "yt_bee6nXyzJ_E_UgxafhxLGL5FhoPfTGJ4AaABAg",
    "yt_afYzk6ojzZU_UgzdkWlIB8VRd-U-dDJ4AaABAg",
    "yt_duZWizK5bxk_Ugy9Dt4XHW_JkxETu914AaABAg",

    # subtip gresit
    "yt_xSxPpRcYQp0_UgzoragxSzrTVcnWjyZ4AaABAg",
    "yt_xSxPpRcYQp0_UgzyMkDmRgKi-psiY694AaABAg",
    "yt_jvV4hfEOyQM_UgzSXKCS41EUisyh7mR4AaABAg",
    "yt_jvV4hfEOyQM_UgwL3bKMOCBE6HIOM8R4AaABAg",
    "yt_jvV4hfEOyQM_UgzbV1R0UQIKAxjeEeJ4AaABAg",
]

clean_df = (
    my_df[~my_df["id"].isin(REMOVE_IDS)]
    .head(50)
    .copy()
)

clean_df["agent"] = MY_AGENT
clean_df["slug"] = meta["slug"]
clean_df["personality"] = meta["personality"]
clean_df["speaks"] = meta["speaks"]
clean_df["definition"] = meta["definition"]

print("Texte finale:", len(clean_df))
clean_df[["id", "agent", "text"]].head()

Texte finale: 50


,id,agent,text
386,yt_cF6iydQa9ss_UgyMxaKYDJ6_GP9NXXJ4AaABAg,Personalist-salvator,"Eu sper ca va veni si ziua ""Justitiei Divina"" ..."
387,yt_5QYDDejaR5E_UgwOLigdAGsShcgE8Jp4AaABAg,Personalist-salvator,Sunt patrioți și oameni onesti și in interioru...
390,yt_oine49HHTXQ_Ugw57cLmGKlwkomjhG94AaABAg,Personalist-salvator,Doamne ajuta AuR la putere ca SA scapam de hie...
391,yt_oine49HHTXQ_Ugz3uUUbe6bmpTbfJFt4AaABAg,Personalist-salvator,"CUM AU OPRIT EI TURUL 2, AȘA OPRIM ȘI NOI TURU..."
392,yt_HDNif5_foi4_Ugwy-z6hrAqUylVVL3J4AaABAg,Personalist-salvator,Doamne ajută tuturor!!!! Doamne cât râs au făc...


### Descrierea agentului tău
Înainte să exporți bula curată, descrie în 5–7 rânduri ce fel de voce discursivă are agentul ales.
Nu descrie o persoană reală. Descrie un tip de discurs observat în corpus.
Răspunde la aceste întrebări:
- Cum vede acest agent instituțiile, politica sau actorii publici?
- Ce ton folosește cel mai des?
- Ce tip de argumente sau acuzații apar frecvent?
- Ce îl diferențiază de celelalte bule?
- Ce ar trebui să păstreze un viitor agent AI ca să sune coerent cu această bulă?

descriere:
Vocea discursivă a agentului personalist-salvator este construită în jurul ideii că România poate fi „salvată” printr-un lider perceput ca sincer, apropiat de oameni și persecutat de sistem. Instituțiile statului, presa și partidele dominante sunt văzute ca parte a structurii corupte care blochează voința populară și manipulează jocul politic. Tonul este în general emoțional și mobilizator, combină speranța, revolta și apelurile la unitate, deseori într-un registru religios sau moral („Doamne ajută”, „dreptate”, „adevăr”). În locul argumentelor tehnice apar frecvent acuzații legate de corupție, trădare, furt electoral sau influențe externe asupra țării. Față de alte bule, acest tip pune accent mai ales pe relația afectivă dintre lider și „popor”, mai mult decat pe un program politic foarte clar articulat. Agentul AI ar trebui să păstreze limbajul moralizator, ideea de trezire colectivă, opoziția dintre „oamenii simpli” și „sistem”, precum și apelurile constante la unitate, demnitate și recuperarea controlului asupra țării.

In [15]:
# export bula curată pentru etapa următoare

OUT_DIR = Path("data/bubbles")
OUT_DIR.mkdir(parents=True, exist_ok=True)

out_path = OUT_DIR / f"{meta['slug']}.jsonl"

clean_df.to_json(out_path, orient="records", lines=True, force_ascii=False)

print("Salvat:", out_path)

Salvat: data\bubbles\personalist_salvator.jsonl
